In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score
import os, warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {len(tf.config.list_physical_devices('GPU')) > 0}")


In [ ]:
# Change this to your actual dataset name after publishing
DATA = '/kaggle/input/datasets/sangatichapla/mit-ecg-cwt/cwt_output'

X_tr  = np.load(f'{DATA}/X_tr.npy')
X_val = np.load(f'{DATA}/X_val.npy')
X_te  = np.load(f'{DATA}/X_te.npy')
y_tr  = np.load(f'{DATA}/y_tr.npy')
y_val = np.load(f'{DATA}/y_val.npy')
y_te  = np.load(f'{DATA}/y_te.npy')

weights = compute_class_weight('balanced', classes=np.arange(5), y=y_tr)
cw      = dict(enumerate(weights))

CLASS_NAMES = ['Normal','PVC','APB','Paced','Fusion']

print(f"X_tr  : {X_tr.shape}")
print(f"X_val : {X_val.shape}")
print(f"X_te  : {X_te.shape}")
print(f"cw    : {cw}")

In [ ]:
X_tr  = np.transpose(X_tr,  (0, 2, 3, 1))
X_val = np.transpose(X_val, (0, 2, 3, 1))
X_te  = np.transpose(X_te,  (0, 2, 3, 1))

print(f"X_tr  : {X_tr.shape}")
print(f"X_val : {X_val.shape}")
print(f"X_te  : {X_te.shape}")

In [ ]:
# ResNet50V2 pretrained on ImageNet
# Frozen = weights not updated = pure feature extraction
# Output: each scalogram image → 2048-dim feature vector

def extract_resnet(data, batch_size=128):
    base = keras.applications.ResNet50V2(
        include_top=False,
        weights='imagenet',
        input_shape=(64, 64, 3),
        pooling='avg'
    )
    base.trainable = False

    features = []
    for i in range(0, len(data), batch_size):
        batch = keras.applications.resnet_v2.preprocess_input(
                    data[i:i+batch_size] * 255.0)
        feat  = base.predict(batch, verbose=0)
        features.append(feat)
        if (i // batch_size) % 20 == 0:
            print(f"  ResNet: {min(i+batch_size, len(data))}/{len(data)}")
    return np.vstack(features)

print("Extracting ResNet50V2 features...")
print("Train")
R_tr  = extract_resnet(X_tr)
print("Val")
R_val = extract_resnet(X_val)
print("Test")
R_te  = extract_resnet(X_te)

print(f"\nResNet feature shape : {R_tr.shape}")
print(f"Each image → {R_tr.shape[1]} dimensional vector")

In [ ]:
# MobileNetV2 — second pretrained CNN
# Lighter and faster than ResNet
# Output: each image → 1280-dim feature vector

def extract_mobilenet(data, batch_size=128):
    base = keras.applications.MobileNetV2(
        include_top=False,
        weights='imagenet',
        input_shape=(64, 64, 3),
        pooling='avg'
    )
    base.trainable = False

    features = []
    for i in range(0, len(data), batch_size):
        batch = keras.applications.mobilenet_v2.preprocess_input(
                    data[i:i+batch_size] * 255.0)
        feat  = base.predict(batch, verbose=0)
        features.append(feat)
        if (i // batch_size) % 20 == 0:
            print(f"  MobileNet: {min(i+batch_size, len(data))}/{len(data)}")
    return np.vstack(features)

print("Extracting MobileNetV2 features...")
print("Train")
M_tr  = extract_mobilenet(X_tr)
print("Val")
M_val = extract_mobilenet(X_val)
print("Test")
M_te  = extract_mobilenet(X_te)

print(f"\nMobileNet feature shape : {M_tr.shape}")
print(f"Each image → {M_tr.shape[1]} dimensional vector")

# Compare both CNNs
print(f"\nCNN Feature Comparison:")
print(f"  ResNet50V2  → {R_tr.shape[1]} dims per image")
print(f"  MobileNetV2 → {M_tr.shape[1]} dims per image")

In [ ]:
SAVE_DIR = '/kaggle/working/cnn_features'
os.makedirs(SAVE_DIR, exist_ok=True)

np.save(f'{SAVE_DIR}/R_tr.npy',  R_tr)
np.save(f'{SAVE_DIR}/R_val.npy', R_val)
np.save(f'{SAVE_DIR}/R_te.npy',  R_te)
np.save(f'{SAVE_DIR}/M_tr.npy',  M_tr)
np.save(f'{SAVE_DIR}/M_val.npy', M_val)
np.save(f'{SAVE_DIR}/M_te.npy',  M_te)

print("CNN features saved:")
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(f'{SAVE_DIR}/{f}') / 1024**2
    print(f"  {f:<20} {size:.1f} MB")

In [ ]:
def build_finetuned_resnet():
    base = keras.applications.ResNet50V2(
        include_top=False,
        weights='imagenet',
        input_shape=(64, 64, 3),
        pooling='avg'
    )
    # Freeze first 150 layers, unfreeze rest
    for layer in base.layers[:150]:
        layer.trainable = False
    for layer in base.layers[150:]:
        layer.trainable = True

    frozen    = sum(1 for l in base.layers if not l.trainable)
    trainable = sum(1 for l in base.layers if l.trainable)
    print(f"  Frozen layers    : {frozen}")
    print(f"  Trainable layers : {trainable}")

    inputs = keras.Input(shape=(64, 64, 3))
    x      = keras.applications.resnet_v2.preprocess_input(inputs * 255.0)
    x      = base(x, training=False)
    x      = layers.Dense(256, activation='relu')(x)
    x      = layers.Dropout(0.4)(x)
    output = layers.Dense(5, activation='softmax')(x)

    model  = keras.Model(inputs, output)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-5),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print("Fine-tuning ResNet50V2...")
ft_model   = build_finetuned_resnet()
ft_history = ft_model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
    class_weight=cw,
    callbacks=[keras.callbacks.EarlyStopping(
        patience=3, restore_best_weights=True)],
    verbose=1
)

_, ft_acc = ft_model.evaluate(X_te, y_te, verbose=0)
ft_pred   = np.argmax(ft_model.predict(X_te, verbose=0), axis=1)
ft_f1     = f1_score(y_te, ft_pred, average='macro', zero_division=0)

print(f"\nFine-tuned ResNet → Accuracy: {ft_acc:.4f}  F1: {ft_f1:.4f}")
print(classification_report(y_te, ft_pred,
      target_names=CLASS_NAMES, zero_division=0))

In [ ]:

def build_classifier(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(5, activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# ResNet frozen features
print("Training classifier on frozen ResNet features...")
clf_resnet = build_classifier(R_tr.shape[1])
clf_resnet.fit(
    R_tr, y_tr,
    validation_data=(R_val, y_val),
    epochs=20, batch_size=64,
    class_weight=cw,
    callbacks=[keras.callbacks.EarlyStopping(
        patience=5, restore_best_weights=True)],
    verbose=0
)
_, r_acc = clf_resnet.evaluate(R_te, y_te, verbose=0)
r_pred   = np.argmax(clf_resnet.predict(R_te, verbose=0), axis=1)
r_f1     = f1_score(y_te, r_pred, average='macro', zero_division=0)
print(f"Frozen ResNet   → Accuracy: {r_acc:.4f}  F1: {r_f1:.4f}")

# MobileNet frozen features
print("\nTraining classifier on frozen MobileNet features...")
clf_mobile = build_classifier(M_tr.shape[1])
clf_mobile.fit(
    M_tr, y_tr,
    validation_data=(M_val, y_val),
    epochs=20, batch_size=64,
    class_weight=cw,
    callbacks=[keras.callbacks.EarlyStopping(
        patience=5, restore_best_weights=True)],
    verbose=0
)
_, m_acc = clf_mobile.evaluate(M_te, y_te, verbose=0)
m_pred   = np.argmax(clf_mobile.predict(M_te, verbose=0), axis=1)
m_f1     = f1_score(y_te, m_pred, average='macro', zero_division=0)
print(f"Frozen MobileNet → Accuracy: {m_acc:.4f}  F1: {m_f1:.4f}")

In [ ]:
import matplotlib.pyplot as plt

cnn_names = ['ResNet\n(frozen)', 'MobileNet\n(frozen)', 'ResNet\n(finetuned)']
cnn_accs  = [r_acc, m_acc, ft_acc]
cnn_f1s   = [r_f1,  m_f1,  ft_f1]
colors    = ['#2196F3', '#4CAF50', '#FF9800']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(cnn_names, cnn_accs, color=colors)
axes[0].set_title('CNN Accuracy Comparison', fontweight='bold')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0, 1)
for i, v in enumerate(cnn_accs):
    axes[0].text(i, v+0.01, f'{v:.3f}', ha='center', fontsize=10)

axes[1].bar(cnn_names, cnn_f1s, color=colors)
axes[1].set_title('CNN Macro F1 Comparison', fontweight='bold')
axes[1].set_ylabel('Macro F1')
axes[1].set_ylim(0, 1)
for i, v in enumerate(cnn_f1s):
    axes[1].text(i, v+0.01, f'{v:.3f}', ha='center', fontsize=10)

for ax in axes:
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Pretrained CNN Feature Extraction vs Fine-tuning',
             fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCNN Summary:")
print(f"{'Model':<22} {'Accuracy':>10} {'Macro F1':>10}")
print("-" * 44)
for n, a, f in zip(cnn_names, cnn_accs, cnn_f1s):
    print(f"  {n.replace(chr(10),' '):<20} {a:.4f}      {f:.4f}")

print("\nFrozen  = ImageNet features used directly")
print("Finetuned = Last layers retrained on ECG scalograms")
print("Expected: Finetuned > Frozen (adapted to ECG domain)")

In [ ]:
np.save(f'{SAVE_DIR}/ft_acc.npy', np.array([ft_acc]))
np.save(f'{SAVE_DIR}/ft_f1.npy',  np.array([ft_f1]))
np.save(f'{SAVE_DIR}/r_acc.npy',  np.array([r_acc]))
np.save(f'{SAVE_DIR}/r_f1.npy',   np.array([r_f1]))
np.save(f'{SAVE_DIR}/m_acc.npy',  np.array([m_acc]))
np.save(f'{SAVE_DIR}/m_f1.npy',   np.array([m_f1]))

print("All CNN results saved to:", SAVE_DIR)
print("\nPublish this as Kaggle dataset 'mitbih-cnn-features'")
print("Then notebook 3 can load everything from there")
print("\nFiles ready for notebook 3:")
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(f'{SAVE_DIR}/{f}') / 1024**2
    print(f"  {f:<25} {size:.1f} MB")